In [1]:
from google.colab import drive, userdata; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/c2_htr'; RAG = f'{ROOT}/rag'
!pip -q install google-genai rapidfuzz jiwer
import os, sys, json, re, time, joblib, numpy as np, pandas as pd
os.makedirs(f'{ROOT}/code', exist_ok=True); sys.path.append(f'{ROOT}/code')
from google import genai
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print([m.name for m in client.models.list() if 'flash' in m.name])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.8-flash-tts', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-pr

In [2]:
%%writefile /content/drive/MyDrive/c2_htr/code/c2_correction.py
"""Component 2 - uncertainty-scoped, retrieval-grounded LLM correction with a verifier gate."""
import re, json, time, unicodedata
from rapidfuzz.distance import Levenshtein

SIN = re.compile(r'[\u0D80-\u0DFF]'); DIG = re.compile(r'\d')
ID_RE = re.compile(r'l\d+w\d+')                # word ids look like l3w5 (line 3, word 5)
PUNCT = '"\'.,;:!?()[]{}<>|/\\-–—_*#~`“”‘’'
norm = lambda t: unicodedata.normalize('NFC', str(t))
tokens = lambda t: [w.strip(PUNCT) for w in norm(t).split() if w.strip(PUNCT)]
# consonant skeleton: drop vowel signs, virama (්), ං ඃ and the joiner -> the consonants that identify the word.
# Handwriting OCR mostly confuses the small vowel strokes, so R2 compares skeletons.
VOWEL_MARKS = re.compile(r'[\u0DCA-\u0DDF\u0DF2\u0DF3\u0D82\u0D83\u200d]')
skeleton = lambda s: VOWEL_MARKS.sub('', s)

# ---------- 1. uncertainty marking from CRNN character confidences ----------
def words_with_conf(text, confs):
    out, i = [], 0
    for w in text.split(' '):
        c = confs[i:i + len(w)]; i += len(w) + 1
        if w: out.append((w, min(c) if c else 0.0))
    return out

def build_page(lines, tau):
    """lines = [(ocr_text, char_confidences), ...] -> [[{id,text,conf,unsure}, ...], ...]"""
    return [[dict(id=f'l{li}w{wi}', text=w, conf=c, unsure=c < tau)
             for wi, (w, c) in enumerate(words_with_conf(norm(t), conf))]
            for li, (t, conf) in enumerate(lines)]

def render(page, edits=None, marked=False):
    edits = edits or {}
    return '\n'.join(' '.join(
        f"«{w['id']}:{w['text']}»" if marked and w['unsure'] and w['id'] not in edits else edits.get(w['id'], w['text'])
        for w in line) for line in page)

# ---------- 2. prompts ----------
PROMPT_SCOPED = """You fix OCR errors in ONE handwritten civic complaint from Sri Lanka (Sinhala, English or mixed).
Words the OCR was unsure about look like «ID:word» (for example «l2w5:පදුලි» has ID l2w5). Everything else is reliable.
Line breaks only show layout.

RULES
1. Only propose replacements for «ID:word» items. Never touch anything else.
2. Propose a fix only if supported by: a REFERENCE complaint, the LEXICON, the same word elsewhere in this
   complaint, the DESCRIPTION, or the obvious spelling of a common Sinhala/English word.
3. The replacement must look like the OCR word (a few wrong or missing letters). Do not translate,
   summarise or add information. If unsure, leave it out.
4. Never change digits, dates, numbers or codes unless the exact value appears in the evidence.
5. Use the ID exactly as written (e.g. "l2w5"), without « » or the word.
6. Return JSON only: {{"edits": [{{"id": "l2w5", "to": "...", "evidence": "reference|lexicon|complaint|description|spelling"}}]}}

DESCRIPTION (typed by the citizen): {description}

REFERENCE (similar past complaints):
{refs}

LEXICON: {lexicon}

COMPLAINT (OCR):
{text}
{feedback}"""

PROMPT_BLIND = """Correct the OCR errors in this handwritten Sri Lankan civic complaint (Sinhala/English).
Keep the same number of lines. Return JSON only: {{"lines": ["...", "..."]}}
{refs}
OCR TEXT:
{text}"""

def call_llm(client, model, prompt, retries=4):
    for a in range(retries):
        try:
            r = client.models.generate_content(model=model, contents=prompt,
                    config={'temperature': 0, 'response_mime_type': 'application/json'})
            return json.loads(re.search(r'\{.*\}', r.text or '', re.S).group(0))
        except Exception as e:
            if a == retries - 1: raise
            time.sleep(10 * (a + 1))          # rate limit / temporary error -> wait and retry

# ---------- 3. verifier (rules R1-R4) ----------
def check(w, to, vocab, use_verifier=True, max_ratio=0.5):
    if w is None: return ['R1_unknown_id']
    r = []
    if not w['unsure']: r.append('R1_not_uncertain')
    if not to or len(to.split()) > 2: r.append('R1_bad_replacement')
    if use_verifier and not r:
        a, b = skeleton(w['text']), skeleton(to)                                 # R2 on consonant skeletons
        if Levenshtein.distance(a, b) > max(1, max_ratio * max(len(a), len(b))):
            r.append('R2_too_different')
        tk = tokens(to)
        if not tk or any(t not in vocab for t in tk): r.append('R3_unsupported')
        if DIG.findall(w['text']) != DIG.findall(to) and to not in vocab: r.append('R4_digits_changed')
    return r

# ---------- 4. the full loop: mark -> LLM edit list -> verify -> re-check ----------
def correct_page(lines, client, model, refs, lexicon, tau=0.8, use_verifier=True, max_rounds=3,
                 description='', sleep=4):
    page = build_page(lines, tau)
    by_id = {w['id']: w for line in page for w in line}
    raw = render(page)
    base = dict(raw=raw, refs=refs, edits=[], prompts=[], rounds=0)
    if not any(w['unsure'] for w in by_id.values()):
        return {**base, 'final': raw, 'decision': 'NO_CHANGES'}
    vocab = (set(lexicon) | {w['text'] for w in by_id.values() if not w['unsure']}
             | {t for d in refs for t in tokens(d)} | set(tokens(description)))
    lex_txt = ', '.join(lexicon) if lexicon else '(none)'
    refs_txt = '\n'.join(f'- {d}' for d in refs) or '(none)'
    accepted, log, prompts, feedback, rnd = {}, [], [], '', 0
    for rnd in range(1, max_rounds + 1):
        prompt = PROMPT_SCOPED.format(description=description or '(none)', refs=refs_txt, lexicon=lex_txt,
                                      text=render(page, accepted, marked=True),
                                      feedback=f'\nFEEDBACK ON YOUR PREVIOUS EDITS (fix or drop):\n{feedback}' if feedback else '')
        prompts.append(prompt)
        try:
            out = call_llm(client, model, prompt); time.sleep(sleep)
        except Exception as e:
            return {**base, 'final': raw, 'edits': log, 'prompts': prompts, 'rounds': rnd,
                    'decision': 'FALLBACK_RAW', 'error': str(e)}
        rej, n = [], 0
        for e in out.get('edits', []):
            m = ID_RE.search(str(e.get('id', ''))); eid = m.group(0) if m else str(e.get('id', ''))
            to = norm(e.get('to', '')).strip(); w = by_id.get(eid)
            if eid in accepted or (w and to == w['text']): continue
            reasons = check(w, to, vocab, use_verifier); n += 1
            log.append(dict(id=eid, frm=w['text'] if w else '', to=to, evidence=e.get('evidence', ''),
                            ok=not reasons, reasons=reasons, round=rnd))
            if reasons: rej.append(log[-1])
            else: accepted[eid] = to
        if not use_verifier or not rej or len(rej) / max(1, n) <= 0.3: break
        feedback = '\n'.join(f"- {r['id']}: '{r['frm']}' -> '{r['to']}' rejected ({', '.join(r['reasons'])})" for r in rej)
    nrej = sum(not l['ok'] for l in log)
    decision = ('ACCEPT' if accepted and not nrej else 'PARTIAL' if accepted else 'REVIEW' if nrej else 'NO_CHANGES')
    return {**base, 'final': render(page, accepted), 'edits': log, 'prompts': prompts, 'rounds': rnd, 'decision': decision}

# ---------- 5. ablation baseline: LLM rewrites everything (no marks, no verifier) ----------
def correct_blind(lines, client, model, refs=()):
    raw = '\n'.join(norm(t) for t, _ in lines)
    refs_txt = ('REFERENCE (similar past complaints):\n' + '\n'.join(f'- {d}' for d in refs) + '\n') if refs else ''
    try:
        out = call_llm(client, model, PROMPT_BLIND.format(refs=refs_txt, text=raw))
        return dict(raw=raw, final='\n'.join(norm(x) for x in out.get('lines', [])), decision='BLIND')
    except Exception as e:
        return dict(raw=raw, final=raw, decision='FALLBACK_RAW', error=str(e))

Overwriting /content/drive/MyDrive/c2_htr/code/c2_correction.py


In [2]:
import importlib, c2_correction as C; importlib.reload(C)
from rapidfuzz import fuzz
MODEL = 'models/gemini-3.5-flash'          # <- from the Step 1 list

pr = pd.read_csv(f'{ROOT}/test_predictions.csv', keep_default_na=False)
pr['conf'] = pr.conf.map(json.loads)
pr['page'] = pr.file.str.replace(r'_c\d+_l\d+\.png$', '', regex=True)
pr = pr.sort_values('file')                               # reading order: column, then line

docs = [json.loads(l) for l in open(f'{RAG}/past_complaints.jsonl', encoding='utf-8')]
texts = [d['text'] for d in docs]
tfv, TF = joblib.load(f'{RAG}/tfidf.joblib')
lexicon = open(f'{RAG}/lexicon_si.txt', encoding='utf-8').read().split('\n')
from sklearn.metrics.pairwise import linear_kernel

def retrieve(query, truth_lines=(), k=20):
    leak = {i for i, t in enumerate(texts) if C.SIN.search(t) and any(fuzz.ratio(t, tl) >= 90 for tl in truth_lines)}
    s = linear_kernel(tfv.transform([query]), TF).ravel()
    return [texts[i] for i in s.argsort()[::-1][:k + len(leak)] if i not in leak][:k]

pages = {p: g for p, g in pr.groupby('page')}
print({s: pr[pr.split == s].page.nunique() for s in ['val', 'test']}, 'pages | lexicon', len(lexicon))

{'val': 4, 'test': 3} pages | lexicon 583


In [9]:
import importlib, hashlib, difflib, jiwer
importlib.reload(C)                                  # fresh module, then add the cache once
CACHE_F = f'{ROOT}/llm_cache.jsonl'; cache = {}
if os.path.exists(CACHE_F):
    for l in open(CACHE_F, encoding='utf-8'): d = json.loads(l); cache[d['k']] = d['v']
_orig_call = C.call_llm
def cached_call(client, model, prompt, retries=4):
    k = hashlib.sha256((model + prompt).encode('utf-8')).hexdigest()
    if k not in cache:
        cache[k] = _orig_call(client, model, prompt, retries)
        with open(CACHE_F, 'a', encoding='utf-8') as f:
            f.write(json.dumps({'k': k, 'model': model, 'prompt': prompt, 'v': cache[k]}, ensure_ascii=False) + '\n')
    return cache[k]
C.call_llm = cached_call
TAU = 0.8
print('cached answers so far:', len(cache), '| model:', MODEL)

cached answers so far: 5 | model: models/gemini-3.5-flash


In [10]:
CONFIGS = {
    'A. OCR only (CRNN)':                 None,
    'B. Blind LLM rewrite':               'blind',
    'C. Blind LLM + RAG':                 'blind_rag',
    'D. Scoped LLM (marks only)':         'scoped',
    'E. Scoped LLM + RAG + lexicon':      'scoped_rag',
    'F. Full: scoped + RAG + verifier':   'full',
}

def run(cfg, g):
    lines, truth = list(zip(g.pred, g.conf)), list(g.truth)
    refs = retrieve(' '.join(g.pred), truth)                      # leak-filtered top-20
    if cfg is None:          return '\n'.join(g.pred), []
    if cfg == 'blind':       return C.correct_blind(lines, client, MODEL)['final'], []
    if cfg == 'blind_rag':   return C.correct_blind(lines, client, MODEL, refs)['final'], []
    if cfg == 'scoped':      o = C.correct_page(lines, client, MODEL, [], [], tau=TAU, use_verifier=False)
    elif cfg == 'scoped_rag':o = C.correct_page(lines, client, MODEL, refs, lexicon, tau=TAU, use_verifier=False)
    else:                    o = C.correct_page(lines, client, MODEL, refs, lexicon, tau=TAU, use_verifier=True)
    return o['final'], o['edits']

def correct_positions(ref, hyp):
    ok = set()
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False).get_opcodes():
        if tag == 'equal': ok.update(range(i1, i2))
    return ok

In [11]:
SPLIT = 'test'                     # run once with 'test', then once with 'val'
rows, examples = [], []
for p, g in pages.items():
    if g.split.iloc[0] != SPLIT: continue
    truth = ' '.join(g.truth); ref_w = C.tokens(truth)
    raw = ' '.join(g.pred); ok_raw = correct_positions(ref_w, C.tokens(raw))
    for name, cfg in CONFIGS.items():
        final, edits = run(cfg, g)
        fin = ' '.join(final.split('\n')); ok_fin = correct_positions(ref_w, C.tokens(fin))
        suspicious = cfg is not None and cfg.startswith('blind') and fin.strip() == raw.strip()
        rows.append(dict(config=name, page=p, chars=len(truth),
                         cer=jiwer.cer(truth, fin), wer=jiwer.wer(truth, fin),
                         c2w=len(ok_raw - ok_fin), w2c=len(ok_fin - ok_raw),
                         n_ok_raw=len(ok_raw), n_wrong_raw=len(ref_w) - len(ok_raw),
                         kept=sum(e['ok'] for e in edits), reverted=sum(not e['ok'] for e in edits),
                         suspicious=suspicious))
        examples.append(dict(config=name, page=p, truth=truth, raw=raw, final=fin))
        print(f'{p[-12:]} | {name:34s} CER {rows[-1]["cer"]:.3f}  broke {rows[-1]["c2w"]:2d}  fixed {rows[-1]["w2c"]:2d}'
              + ('   !! probably a failed call' if suspicious else ''))

df = pd.DataFrame(rows)
if df.suspicious.any():
    print('\n!! Some blind calls look failed - wait a minute and re-run this cell before using the table.')
agg = df.groupby('config', sort=False)[df.columns.drop('config')].apply(lambda d: pd.Series({
    'CER': (d.cer * d.chars).sum() / d.chars.sum(),
    'WER': d.wer.mean(),
    'correct→wrong (rate)': d.c2w.sum() / max(1, d.n_ok_raw.sum()),
    'wrong→correct (rate)': d.w2c.sum() / max(1, d.n_wrong_raw.sum()),
    'edits kept': d.kept.sum(), 'edits reverted': d.reverted.sum()}))
print('\n', agg.round(3).to_string())
agg.round(4).to_csv(f'{ROOT}/ablation_{SPLIT}.csv')
json.dump(examples, open(f'{ROOT}/ablation_{SPLIT}_examples.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

6_10_07_p033 | A. OCR only (CRNN)                 CER 0.261  broke  0  fixed  0
6_10_07_p033 | B. Blind LLM rewrite               CER 0.243  broke  0  fixed 12
6_10_07_p033 | C. Blind LLM + RAG                 CER 0.196  broke  0  fixed 12
6_10_07_p033 | D. Scoped LLM (marks only)         CER 0.246  broke  0  fixed  6
6_10_07_p033 | E. Scoped LLM + RAG + lexicon      CER 0.243  broke  0  fixed  4
6_10_07_p033 | F. Full: scoped + RAG + verifier   CER 0.243  broke  0  fixed  4
6_10_07_p048 | A. OCR only (CRNN)                 CER 0.286  broke  0  fixed  0
6_10_07_p048 | B. Blind LLM rewrite               CER 0.312  broke  2  fixed 11
6_10_07_p048 | C. Blind LLM + RAG                 CER 0.278  broke  2  fixed 14
6_10_07_p048 | D. Scoped LLM (marks only)         CER 0.263  broke  0  fixed  3
6_10_07_p048 | E. Scoped LLM + RAG + lexicon      CER 0.248  broke  0  fixed  4
6_10_07_p048 | F. Full: scoped + RAG + verifier   CER 0.248  broke  0  fixed  4
6_10_07_p068 | A. OCR only (CRNN)       

In [6]:
g = pages['CamScanner_10-02-2026_10_07_p068']
o = C.correct_blind(list(zip(g.pred, g.conf)), client, MODEL)
print(o['decision'], '|', o.get('error', 'no error')[:300])

FALLBACK_RAW | 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 


In [8]:
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY_2'))
r = client.models.generate_content(model=MODEL, contents='Reply with JSON {"ok": true}',
                                   config={'temperature': 0, 'response_mime_type': 'application/json'})
print('key works:', r.text.strip())

key works: {"ok": true}


In [12]:
def verify_rewrite(g, rewritten):
    lines = list(zip(g.pred, g.conf))
    page = C.build_page(lines, TAU); words = [w for line in page for w in line]
    raw_t, new_t = [w['text'] for w in words], C.norm(rewritten).split()
    refs = retrieve(' '.join(g.pred), list(g.truth))
    vocab = set(lexicon) | {w['text'] for w in words if not w['unsure']} | {t for d in refs for t in C.tokens(d)}
    out, kept, rev = [], 0, 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=raw_t, b=new_t, autojunk=False).get_opcodes():
        if tag == 'equal':
            out += raw_t[i1:i2]
        elif tag == 'replace' and i2 - i1 == j2 - j1:          # word-for-word changes -> check each
            for k in range(i2 - i1):
                w, to = words[i1 + k], new_t[j1 + k]
                if C.check(w, to, vocab): out.append(w['text']); rev += 1      # failed a rule -> revert
                else: out.append(to); kept += 1
        else:                                                    # added / removed / merged words -> revert
            out += raw_t[i1:i2]; rev += max(i2 - i1, j2 - j1)
    return ' '.join(out), kept, rev

c_out = {e['page']: e['final'] for e in examples if e['config'].startswith('C.')}
for p, g in pages.items():
    if g.split.iloc[0] != SPLIT or p not in c_out: continue
    truth = ' '.join(g.truth); ref_w = C.tokens(truth)
    raw = ' '.join(g.pred); ok_raw = correct_positions(ref_w, C.tokens(raw))
    fin, kept, rev = verify_rewrite(g, c_out[p]); ok_fin = correct_positions(ref_w, C.tokens(fin))
    rows.append(dict(config='G. Blind+RAG rewrite + verifier', page=p, chars=len(truth),
                     cer=jiwer.cer(truth, fin), wer=jiwer.wer(truth, fin), c2w=len(ok_raw - ok_fin),
                     w2c=len(ok_fin - ok_raw), n_ok_raw=len(ok_raw), n_wrong_raw=len(ref_w) - len(ok_raw),
                     kept=kept, reverted=rev, suspicious=False))
    print(f'{p[-12:]} | G  CER {rows[-1]["cer"]:.3f}  broke {rows[-1]["c2w"]}  fixed {rows[-1]["w2c"]}  kept {kept} reverted {rev}')

df = pd.DataFrame(rows)
agg = df.groupby('config', sort=False)[df.columns.drop('config')].apply(lambda d: pd.Series({
    'CER': (d.cer * d.chars).sum() / d.chars.sum(), 'WER': d.wer.mean(),
    'correct→wrong (rate)': d.c2w.sum() / max(1, d.n_ok_raw.sum()),
    'wrong→correct (rate)': d.w2c.sum() / max(1, d.n_wrong_raw.sum()),
    'edits kept': d.kept.sum(), 'edits reverted': d.reverted.sum()}))
print('\n', agg.round(3).to_string()); agg.round(4).to_csv(f'{ROOT}/ablation_{SPLIT}.csv')

6_10_07_p033 | G  CER 0.239  broke 0  fixed 5  kept 5 reverted 28
6_10_07_p048 | G  CER 0.259  broke 0  fixed 3  kept 9 reverted 23
6_10_07_p068 | G  CER 0.280  broke 0  fixed 3  kept 3 reverted 39

                                     CER    WER  correct→wrong (rate)  wrong→correct (rate)  edits kept  edits reverted
config                                                                                                                
A. OCR only (CRNN)                0.282  0.766                 0.000                 0.000         0.0             0.0
B. Blind LLM rewrite              0.212  0.481                 0.143                 0.457         0.0             0.0
C. Blind LLM + RAG                0.200  0.424                 0.054                 0.500         0.0             0.0
D. Scoped LLM (marks only)        0.260  0.670                 0.018                 0.196        22.0             0.0
E. Scoped LLM + RAG + lexicon     0.259  0.674                 0.000                 0

In [13]:
truth_words = {p: set(C.tokens(' '.join(g.truth))) for p, g in pages.items() if g.split.iloc[0] == SPLIT}
stats = collections.Counter() if 'collections' in dir() else None
import collections; stats = collections.Counter()
for p, g in pages.items():
    if g.split.iloc[0] != SPLIT or p not in c_out: continue
    page = C.build_page(list(zip(g.pred, g.conf)), TAU); words = [w for l in page for w in l]
    raw_t, new_t = [w['text'] for w in words], C.norm(c_out[p]).split()
    refs = retrieve(' '.join(g.pred), list(g.truth))
    vocab = set(lexicon) | {w['text'] for w in words if not w['unsure']} | {t for d in refs for t in C.tokens(d)}
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=raw_t, b=new_t, autojunk=False).get_opcodes():
        if tag == 'equal': continue
        if tag == 'replace' and i2 - i1 == j2 - j1:
            for k in range(i2 - i1):
                to = new_t[j1 + k]; good = to.strip(C.PUNCT) in truth_words[p]
                for r in (C.check(words[i1 + k], to, vocab) or ['kept']):
                    stats[(r.split('_')[0] if r != 'kept' else 'kept', 'correct fix' if good else 'wrong fix')] += 1
        else:
            stats[('merge/split/insert', 'n/a')] += max(i2 - i1, j2 - j1)
print(pd.Series(stats).unstack(fill_value=0))

                    correct fix  n/a  wrong fix
R1                            6    0          4
R2                           10    0          5
R3                            8    0          6
R4                            1    0          1
kept                         13    0          4
merge/split/insert            0   53          0
